# Deep Learning PR 2 - Adult Income Dataset

I have kept the code simple and beginner friendly.

We are going to predict whether a person's income is **<=50K or >50K** using an ANN.


## Task 1: Data Loading, Cleaning & Exploratory Data Analysis

### Task 1.1 - Load and inspect the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("adult.csv")

print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())

# remove extra spaces from text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

print("\nColumns:")
print(df.columns.tolist())

### Task 1.2 - Handle missing values

`?` is treated as a missing value. The missing rows are removed because the missing columns are categorical and simple mode filling could make the most common category even bigger. On this copy of the Adult dataset, dropping rows with any missing value leaves **45,222 rows** (the number can differ from a handout if its expected count uses a different cleaning rule).

In [ ]:
df = df.replace("?", np.nan)

print("Missing values:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

# remove rows which have any missing value
df = df.dropna()

print("\nShape after removing missing rows:", df.shape)
print("Missing values now:", df.isnull().sum().sum())

### Task 1.3 - Drop redundant columns

`fnlwgt` is a census sampling weight, so it is not used as a predictive feature. `education` is also dropped because `educational-num` already gives the same education information in numeric form.

In [ ]:
df = df.drop(columns=["fnlwgt", "education"])

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

### Task 1.4 - Encode the target

In [ ]:
df["income"] = (df["income"] == ">50K").astype(int)

print(df["income"].value_counts())
print("\nClass percentages:")
print(df["income"].value_counts(normalize=True) * 100)

ax = sns.countplot(data=df, x="income")
plt.title("Income Class Count")
plt.xlabel("Income (0 = <=50K, 1 = >50K)")
plt.ylabel("Count")

for p in ax.patches:
    ax.annotate(int(p.get_height()),
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")

plt.show()

**Important:** Class 0 is much larger than class 1. So accuracy alone is not enough. We will also check Precision, Recall and F1-score, especially for class 1.

### Task 1.5 - Exploratory visualisations

In [ ]:
# Age coloured by income
sns.histplot(data=df, x="age", hue="income", bins=30, kde=True)
plt.title("Age by Income")
plt.show()

# Hours per week by income
sns.boxplot(data=df, x="income", y="hours-per-week")
plt.title("Hours per Week by Income")
plt.show()

# Education number by income
sns.boxplot(data=df, x="income", y="educational-num")
plt.title("Education Number by Income")
plt.show()

# Correlation heatmap
plt.figure(figsize=(10, 7))
corr = df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

The plots help us see that variables such as age, education and hours worked can contain useful information for predicting income. Correlation only shows linear relationships, so the ANN can also learn non-linear patterns.

### Task 1.6 - Encode categoricals and scale numerics

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

X = df.drop(columns=["income"])
y = df["income"]

categorical_cols = X.select_dtypes(include="object").columns.tolist()
numeric_cols = X.select_dtypes(exclude="object").columns.tolist()

# split first so test data is not used while fitting preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
        ("num", StandardScaler(), numeric_cols)
    ]
)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nClass ratio in train:")
print(y_train.value_counts(normalize=True))

print("\nClass ratio in test:")
print(y_test.value_counts(normalize=True))

input_dim = X_train.shape[1]
print("\nInput dimension:", input_dim)

## Task 2: Baseline ANN

### Task 2.1 - Define reusable `build_ann()` function

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def build_ann(input_dim, hidden_units=[128, 64],
              activation="relu",
              initializer="glorot_uniform",
              use_batchnorm=False,
              optimizer="adam",
              loss="binary_crossentropy"):

    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for units in hidden_units:
        model.add(layers.Dense(units, activation=activation,
                               kernel_initializer=initializer))
        if use_batchnorm:
            model.add(layers.BatchNormalization())
            model.add(layers.Activation(activation))

    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=["accuracy"]
    )

    return model

### Task 2.2 - Baseline ANN architecture

In [ ]:
model_base = build_ann(
    input_dim=input_dim,
    hidden_units=[128, 64],
    activation="relu",
    initializer="glorot_uniform",
    optimizer="adam",
    loss="binary_crossentropy"
)

model_base.summary()

# Parameters:
# first layer = input_dim*128 + 128
# second layer = 128*64 + 64
# output layer = 64*1 + 1

print("Input dimension:", input_dim)

The output layer has one sigmoid neuron because this is binary classification. Sigmoid converts the output into a value between 0 and 1, which can be treated as the probability of class 1.

### Task 2.3 - Train the baseline

In [ ]:
history_base = model_base.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history_base.history["loss"], label="train")
plt.plot(history_base.history["val_loss"], label="validation")
plt.title("Baseline ANN - Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_base.history["accuracy"], label="train")
plt.plot(history_base.history["val_accuracy"], label="validation")
plt.title("Baseline ANN - Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

### Task 2.4 - Evaluate the baseline with full metrics

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

y_pred_prob = model_base.predict(X_test).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["<=50K", ">50K"],
            yticklabels=["<=50K", ">50K"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Baseline Confusion Matrix")
plt.show()

precision_1 = precision_score(y_test, y_pred, pos_label=1)
recall_1 = recall_score(y_test, y_pred, pos_label=1)
f1_1 = f1_score(y_test, y_pred, pos_label=1)
auc = roc_auc_score(y_test, y_pred_prob)

print("Precision class 1:", precision_1)
print("Recall class 1:", recall_1)
print("F1 class 1:", f1_1)
print("ROC-AUC:", auc)

fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Baseline ROC Curve")
plt.legend()
plt.show()

### Task 2.5 - WHY tabular data suits ANN

Tabular data contains rows and columns with numeric and categorical information. After one-hot encoding and scaling, every input becomes a numeric value. ANN hidden layers can learn combinations such as age + education + hours worked. This can capture patterns that a simple linear model may miss.

## Task 3: Activation Function Experiments

### Task 3.1 - Train 4 models with different hidden activations

In [ ]:
activations = ["relu", "tanh", "sigmoid", "elu"]

histories = {}
models_activation = {}
activation_results = []

for act in activations:
    print("\nTraining:", act)

    model = build_ann(
        input_dim=input_dim,
        hidden_units=[128, 64],
        activation=act,
        initializer="glorot_uniform",
        optimizer="adam",
        loss="binary_crossentropy"
    )

    history = model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=64,
        validation_split=0.1,
        verbose=0
    )

    models_activation[act] = model
    histories[act] = history

    prob = model.predict(X_test, verbose=0).flatten()
    pred = (prob > 0.5).astype(int)
    activation_results.append({
        "Activation": act,
        "Val Accuracy": history.history["val_accuracy"][-1],
        "F1 class 1": f1_score(y_test, pred),
        "Recall class 1": recall_score(y_test, pred)
    })

activation_results_df = pd.DataFrame(activation_results)
display(activation_results_df)

### Task 3.2 - Activation comparison 4-panel plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, act in zip(axes.ravel(), activations):
    h = histories[act]
    ax.plot(h.history["accuracy"], label="train")
    ax.plot(h.history["val_accuracy"], label="validation")

    final_acc = h.history["val_accuracy"][-1]
    pred = (models_activation[act].predict(X_test, verbose=0).flatten() > 0.5).astype(int)
    final_f1 = f1_score(y_test, pred)

    ax.set_title(f"{act} | Val Acc={final_acc:.3f} | F1={final_f1:.3f}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.legend()

plt.tight_layout()
plt.show()

### Task 3.3 - Dead neuron check for ReLU

In [ ]:
relu_model = models_activation["relu"]

# take the first hidden layer
intermediate = keras.Model(
    inputs=relu_model.input,
    outputs=relu_model.layers[0].output
)

activations_relu = intermediate.predict(X_test[:500], verbose=0)

dead_fraction = np.mean(activations_relu == 0, axis=0)

print("Dead fraction for first hidden layer:")
print(dead_fraction)

plt.figure(figsize=(10, 4))
plt.hist(dead_fraction, bins=20)
plt.xlabel("Fraction of zero outputs")
plt.ylabel("Number of neurons")
plt.title("ReLU First-Layer Dead Neuron Distribution")
plt.show()

ReLU outputs zero for negative inputs. A neuron with zero output for almost all samples may become inactive, which is called a dead neuron. ReLU is still popular because it is simple and usually gives fast training.

### Task 3.4 - Gradient flow check - sigmoid hidden layers

In [ ]:
sigmoid_model = models_activation["sigmoid"]

x_small = tf.convert_to_tensor(X_train[:256], dtype=tf.float32)
y_small = tf.convert_to_tensor(np.array(y_train.iloc[:256]).reshape(-1, 1), dtype=tf.float32)

with tf.GradientTape() as tape:
    pred_small = sigmoid_model(x_small, training=True)
    loss_value = tf.reduce_mean(
        keras.losses.binary_crossentropy(y_small, pred_small)
    )

grads = tape.gradient(loss_value, sigmoid_model.trainable_weights)

gradient_values = []
gradient_names = []

for w, g in zip(sigmoid_model.trainable_weights, grads):
    if g is not None:
        gradient_values.append(tf.reduce_mean(tf.abs(g)).numpy())
        gradient_names.append(w.name)

print("Average absolute gradient:")
for name, value in zip(gradient_names, gradient_values):
    print(name, ":", value)

plt.figure(figsize=(10, 4))
plt.bar(range(len(gradient_values)), gradient_values)
plt.xticks(range(len(gradient_values)), gradient_names, rotation=90)
plt.yscale("log")
plt.ylabel("Mean absolute gradient")
plt.title("Gradient Magnitude by Layer - Sigmoid ANN")
plt.tight_layout()
plt.show()

Sigmoid has derivative `σ(x)(1-σ(x))`, whose maximum value is 0.25. When sigmoid becomes saturated, its derivative becomes very small. Across many layers these small gradients can multiply together, causing vanishing gradients.

### Task 3.5 - Activation summary table

In [ ]:
activation_summary = pd.DataFrame({
    "Activation": ["ReLU", "Tanh", "Sigmoid", "ELU"],
    "Formula": [
        "max(0, x)",
        "tanh(x)",
        "1 / (1 + exp(-x))",
        "x if x>0 else alpha*(exp(x)-1)"
    ],
    "Output Range": [
        "[0, infinity)",
        "(-1, 1)",
        "(0, 1)",
        "(-alpha, infinity)"
    ],
    "Zero-Centered": ["No", "Yes", "No", "Mostly no"],
    "Vanishing Gradient Risk": ["Low/medium", "Medium", "High", "Low/medium"],
    "Dead Neuron Risk": ["Yes", "No", "No", "Less than ReLU"],
    "Main Use": [
        "Hidden layers",
        "Some hidden layers",
        "Output for binary classification",
        "Hidden layers"
    ]
})

display(activation_summary)

## Task 4: Weight Initialization Techniques

### Task 4.1 - Train 5 models with different initializers

In [ ]:
initializers = ["glorot_uniform", "glorot_normal", "he_uniform", "he_normal", "zeros"]

histories_init = {}
models_init = {}
init_results = []

for init in initializers:
    print("\nTraining:", init)

    model = build_ann(
        input_dim=input_dim,
        hidden_units=[128, 64],
        activation="relu",
        initializer=init,
        optimizer="adam",
        loss="binary_crossentropy"
    )

    history = model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=64,
        validation_split=0.1,
        verbose=0
    )

    histories_init[init] = history
    models_init[init] = model

    prob = model.predict(X_test, verbose=0).flatten()
    pred = (prob > 0.5).astype(int)

    init_results.append({
        "Initializer": init,
        "Final Val Accuracy": history.history["val_accuracy"][-1],
        "F1 class 1": f1_score(y_test, pred)
    })

init_results_df = pd.DataFrame(init_results)
display(init_results_df)

### Task 4.2 - WHY initialisation matters

Neural networks should not start with all weights equal. If all weights are zero, neurons in the same layer produce the same output, get the same gradient and keep updating in the same way. This symmetry problem makes the network behave like only one effective neuron. Glorot initialization keeps the variance suitable for the layer size. He initialization is designed especially for ReLU and uses a larger variance to compensate for ReLU setting negative values to zero.

### Task 4.3 - Convergence speed comparison

In [ ]:
plt.figure(figsize=(10, 6))

for init in initializers:
    plt.plot(
        histories_init[init].history["val_accuracy"],
        label=init
    )

plt.axhline(0.84, linestyle="--", label="0.84 target")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Weight Initialization - Convergence Speed Comparison")
plt.legend()
plt.show()

print("Final validation accuracy:")
for init in initializers:
    print(init, ":", histories_init[init].history["val_accuracy"][-1])

The zero initializer is expected to perform badly because of the symmetry problem. He initialization is generally a good choice for ReLU networks.

### Task 4.4 - Zeros failure demonstration

In [ ]:
zero_history = histories_init["zeros"]

plt.figure(figsize=(10, 5))
plt.plot(zero_history.history["loss"], label="training loss")
plt.plot(zero_history.history["val_accuracy"], label="validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.title("Zeros Initialization - Failure Demonstration")
plt.legend()
plt.show()

print("Final zero-init validation accuracy:",
      zero_history.history["val_accuracy"][-1])

With zero initialization, all neurons start identically and receive identical updates. The model cannot properly break symmetry. Because class 0 is the majority class, a nearly constant prediction can still give accuracy around the majority-class percentage, but it does not mean the model has learned useful features.

### Task 4.5 - Weight distribution visualisation

In [ ]:
# initial weights from fresh models
he_model = build_ann(input_dim, [128, 64], "relu", "he_normal")
glorot_model = build_ann(input_dim, [128, 64], "relu", "glorot_uniform")

he_initial = he_model.layers[0].get_weights()[0].flatten()
glorot_initial = glorot_model.layers[0].get_weights()[0].flatten()

# weights after training
he_trained = models_init["he_normal"].layers[0].get_weights()[0].flatten()
glorot_trained = models_init["glorot_uniform"].layers[0].get_weights()[0].flatten()

plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.hist(he_initial, bins=40)
plt.title("He Normal - Before Training")

plt.subplot(2, 2, 2)
plt.hist(glorot_initial, bins=40)
plt.title("Glorot Uniform - Before Training")

plt.subplot(2, 2, 3)
plt.hist(he_trained, bins=40)
plt.title("He Normal - After Training")

plt.subplot(2, 2, 4)
plt.hist(glorot_trained, bins=40)
plt.title("Glorot Uniform - After Training")

plt.tight_layout()
plt.show()

## Task 5: Loss Functions

### Task 5.1 - Binary Cross-Entropy

For binary classification, Binary Cross-Entropy is:

**L = -[y log(p) + (1-y) log(1-p)]**

Here `y` is the true label and `p` is the predicted probability. It strongly penalizes confident wrong predictions, so it is a natural loss for a sigmoid binary classifier.

### Task 5.2 - MSE as a classification loss

In [ ]:
model_mse = build_ann(
    input_dim=input_dim,
    hidden_units=[128, 64],
    activation="relu",
    initializer="he_normal",
    optimizer="adam",
    loss="mean_squared_error"
)

history_mse = model_mse.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    verbose=0
)

plt.figure(figsize=(10, 5))
plt.plot(history_base.history["val_loss"], label="BCE")
plt.plot(history_mse.history["val_loss"], label="MSE")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("BCE vs MSE as Classification Loss")
plt.legend()
plt.show()

prob_mse = model_mse.predict(X_test, verbose=0).flatten()
pred_mse = (prob_mse > 0.5).astype(int)

print("MSE Accuracy:", (pred_mse == y_test).mean())
print("MSE F1 class 1:", f1_score(y_test, pred_mse))

MSE can work for classification, but BCE is normally better for a sigmoid output because BCE gives a stronger penalty to confident wrong probabilities and matches the Bernoulli classification problem.

### Task 5.3 - Weighted Binary Cross-Entropy for class imbalance

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight = {0: weights[0], 1: weights[1]}
print("Class weights:", class_weight)

model_wbce = build_ann(
    input_dim=input_dim,
    hidden_units=[128, 64],
    activation="relu",
    initializer="he_normal",
    optimizer="adam",
    loss="binary_crossentropy"
)

history_wbce = model_wbce.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    class_weight=class_weight,
    verbose=0
)

prob_wbce = model_wbce.predict(X_test, verbose=0).flatten()
pred_wbce = (prob_wbce > 0.5).astype(int)

print(classification_report(y_test, pred_wbce))

print("Precision class 1:", precision_score(y_test, pred_wbce))
print("Recall class 1:", recall_score(y_test, pred_wbce))
print("F1 class 1:", f1_score(y_test, pred_wbce))

Weighted BCE gives more importance to the minority class. This usually improves recall for class 1, although precision can decrease because the model is being encouraged to find more positive cases.

### Task 5.4 - Focal Loss

In [ ]:
def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)

        bce = -(y_true * tf.math.log(y_pred) +
                (1 - y_true) * tf.math.log(1 - y_pred))

        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        weight = alpha * y_true + (1 - alpha) * (1 - y_true)

        focal = weight * tf.pow(1 - p_t, gamma) * bce
        return tf.reduce_mean(focal)

    return loss

model_focal = build_ann(
    input_dim=input_dim,
    hidden_units=[128, 64],
    activation="relu",
    initializer="he_normal",
    optimizer="adam",
    loss=focal_loss(gamma=2.0, alpha=0.25)
)

history_focal = model_focal.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    verbose=0
)

prob_focal = model_focal.predict(X_test, verbose=0).flatten()
pred_focal = (prob_focal > 0.5).astype(int)

print(classification_report(y_test, pred_focal))
print("Focal F1 class 1:", f1_score(y_test, pred_focal))

Focal Loss was introduced to reduce the effect of easy examples and focus training more on hard or misclassified examples. This can be useful when the classes are imbalanced.

### Task 5.5 - Loss function summary

In [ ]:
loss_summary = pd.DataFrame({
    "Loss Function": [
        "Binary Cross-Entropy",
        "Weighted BCE",
        "MSE",
        "Focal Loss"
    ],
    "Best For": [
        "Normal binary classification",
        "Imbalanced binary classification",
        "Regression / simple classification experiment",
        "Strong class imbalance / hard examples"
    ],
    "Handles Imbalance": [
        "No",
        "Yes",
        "No",
        "Yes"
    ],
    "Class 1 F1": [
        f1_score(y_test, y_pred),
        f1_score(y_test, pred_wbce),
        f1_score(y_test, pred_mse),
        f1_score(y_test, pred_focal)
    ]
})

display(loss_summary)

print("For this Adult Income problem, Weighted BCE or Focal Loss can be useful when recall of >50K is important.")

## Task 6: Batch Normalization

### Task 6.1 - Build ANN with Batch Normalization

In [ ]:
model_bn = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, kernel_initializer="he_normal"),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    layers.Dense(64, kernel_initializer="he_normal"),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    layers.Dense(1, activation="sigmoid")
])

model_bn.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_bn.summary()

history_bn = model_bn.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    verbose=0
)

### Task 6.2 - WHY Batch Normalization

During training, the input distribution received by later layers can keep changing as earlier weights change. Batch Normalization normalizes layer inputs using the current mini-batch and then uses learnable gamma and beta values. This can make training faster and more stable. At inference time, moving mean and variance collected during training are used.

### Task 6.3 - Train and compare with and without BatchNorm

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history_base.history["val_loss"], label="Without BatchNorm")
plt.plot(history_bn.history["val_loss"], label="With BatchNorm")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Batch Normalization vs Baseline")
plt.legend()
plt.show()

prob_bn = model_bn.predict(X_test, verbose=0).flatten()
pred_bn = (prob_bn > 0.5).astype(int)

print("Baseline Accuracy:", (y_pred == y_test).mean())
print("Baseline F1 class 1:", f1_score(y_test, y_pred))

print("\nBatchNorm Accuracy:", (pred_bn == y_test).mean())
print("BatchNorm F1 class 1:", f1_score(y_test, pred_bn))

### Task 6.4 - BatchNorm position experiment

In [ ]:
# BatchNorm after activation
model_bn_after = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation="relu", kernel_initializer="he_normal"),
    layers.BatchNormalization(),
    layers.Dense(64, activation="relu", kernel_initializer="he_normal"),
    layers.BatchNormalization(),
    layers.Dense(1, activation="sigmoid")
])

# BatchNorm before activation
model_bn_before = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, kernel_initializer="he_normal"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dense(64, kernel_initializer="he_normal"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dense(1, activation="sigmoid")
])

for model in [model_bn_after, model_bn_before]:
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history_bn_after = model_bn_after.fit(
    X_train, y_train, epochs=50, batch_size=64,
    validation_split=0.1, verbose=0
)

history_bn_before = model_bn_before.fit(
    X_train, y_train, epochs=50, batch_size=64,
    validation_split=0.1, verbose=0
)

plt.figure(figsize=(10, 5))
plt.plot(history_bn_after.history["val_accuracy"], label="Dense-ReLU-BN")
plt.plot(history_bn_before.history["val_accuracy"], label="Dense-BN-ReLU")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("BatchNorm Position Comparison")
plt.legend()
plt.show()

In many practical networks, BatchNorm is placed before the activation (`Dense → BN → ReLU`). The exact best order can depend on the architecture and dataset, so this experiment compares both.

### Task 6.5 - Inspect learned gamma and beta

In [ ]:
bn_layer = model_bn.layers[2]  # first BatchNormalization layer

gamma, beta, running_mean, running_var = bn_layer.get_weights()

print("Gamma shape:", gamma.shape)
print("Beta shape:", beta.shape)
print("First 10 gamma values:", gamma[:10])
print("First 10 beta values:", beta[:10])

plt.figure(figsize=(10, 4))
plt.bar(range(len(gamma)), gamma)
plt.xlabel("Neuron")
plt.ylabel("Gamma")
plt.title("Learned Gamma Values")
plt.show()

print("Neurons with |gamma| close to 0:",
      np.sum(np.abs(gamma) < 0.1))

Gamma controls the scale after normalization and beta controls the shift. A gamma close to zero means that feature's normalized signal is being strongly reduced. Larger absolute gamma values mean the model is keeping a stronger signal from that neuron.

## Task 7: Optimizer Comparison

### Task 7.1 - Build 5 models with different optimizers

In [ ]:
optimizers = {
    "SGD": keras.optimizers.SGD(learning_rate=0.01),
    "SGD_Momentum": keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "RMSprop": keras.optimizers.RMSprop(learning_rate=0.001),
    "Adam": keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999),
    "Adamax": keras.optimizers.Adamax(learning_rate=0.001, beta_1=0.9, beta_2=0.999)
}

histories_opt = {}
models_opt = {}
optimizer_results = []

for name, opt in optimizers.items():
    print("\nTraining:", name)

    model = build_ann(
        input_dim=input_dim,
        hidden_units=[128, 64],
        activation="relu",
        initializer="he_normal",
        optimizer=opt,
        loss="binary_crossentropy"
    )

    history = model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=64,
        validation_split=0.1,
        verbose=0
    )

    histories_opt[name] = history
    models_opt[name] = model

    prob = model.predict(X_test, verbose=0).flatten()
    pred = (prob > 0.5).astype(int)

    optimizer_results.append({
        "Optimizer": name,
        "Final Val Accuracy": history.history["val_accuracy"][-1],
        "F1 class 1": f1_score(y_test, pred)
    })

optimizer_results_df = pd.DataFrame(optimizer_results)
display(optimizer_results_df)

### Task 7.2 - Optimizer convergence plot

In [ ]:
plt.figure(figsize=(11, 6))

for name in optimizers:
    plt.plot(
        histories_opt[name].history["val_accuracy"],
        label=name
    )

plt.axhline(0.80, linestyle="--", label="80%")
plt.axhline(0.86, linestyle="--", label="86%")

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Optimizer Comparison - Validation Accuracy")
plt.legend()
plt.show()

print("Final validation accuracies:")
for name in optimizers:
    print(name, ":", histories_opt[name].history["val_accuracy"][-1])

best_optimizer = optimizer_results_df.loc[
    optimizer_results_df["F1 class 1"].idxmax(), "Optimizer"
]
print("\nBest F1 for class 1 in this run:", best_optimizer)

### Task 7.3 - WHY Adam often works well

**SGD:** uses one global learning rate, so convergence can be slower on features with different scales.

**SGD + Momentum:** keeps part of the previous update, which helps move faster in a useful direction.

**RMSprop:** scales updates using a moving average of squared gradients, which can help when gradients have different sizes.

**Adam:** combines momentum (first moment) and RMSprop-like scaling (second moment). It also uses bias correction, so it usually works well without much manual tuning.

**Adamax:** is a stable variant of Adam based on an infinity norm.

For this Adult Income ANN, the optimizer with the highest final F1-score for class 1 is shown by the code above. The exact winner can change slightly because neural network training has randomness.

## Final Results Table

In [ ]:
# Collect the main results in one simple table
final_results = []

def add_result(name, model):
    prob = model.predict(X_test, verbose=0).flatten()
    pred = (prob > 0.5).astype(int)
    final_results.append({
        "Model": name,
        "Accuracy": (pred == y_test).mean(),
        "Precision class 1": precision_score(y_test, pred),
        "Recall class 1": recall_score(y_test, pred),
        "F1 class 1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

add_result("Baseline ANN - BCE", model_base)
add_result("MSE", model_mse)
add_result("Weighted BCE", model_wbce)
add_result("Focal Loss", model_focal)
add_result("BatchNorm", model_bn)

for name, model in models_opt.items():
    add_result("Optimizer - " + name, model)

final_results_df = pd.DataFrame(final_results)
display(final_results_df.sort_values("F1 class 1", ascending=False))

## Conclusion

- The Adult Income dataset has an imbalanced target, so F1 and Recall for class 1 are important.
- ReLU is a strong general activation for hidden layers.
- Sigmoid is useful for the final binary output neuron.
- Zero weight initialization is not suitable for a normal neural network because of the symmetry problem.
- He initialization works well with ReLU.
- BCE is the normal loss for binary classification.
- Weighted BCE and Focal Loss can help when the minority class is important.
- Batch Normalization can make training more stable.
- Adam is usually a strong starting optimizer, but the final result should be based on the actual experiment.

**Note:** Run the notebook from top to bottom because later tasks use models and variables created in earlier tasks.
